# Black–Scholes benchmarks vs Monte Carlo

**Start here:** This deep dive expands on `07_advanced_quant/monte_carlo_simulation.ipynb`; read the overview first for the common `TimeGrid`, `McEngine`, and pricer workflow.

Closed-form **Black–Scholes** prices anchor **European** Monte Carlo. Use them for **put–call parity** checks, **convergence** in **`num_paths`**, and **finite-difference Greeks** as a teaching reference.


## Concept

For GBM with constant volatility, **`black_scholes_call`** / **`black_scholes_put`** implement the standard formulas. **Put–call parity** links calls and puts with the same strike and expiry. **Monte Carlo error** scales like \( 1/\sqrt{N} \) for crude averaging. **Bump-and-revalue** on spot and vol approximates **delta** and **vega**.

## API walkthrough

Import **`black_scholes_call`**, **`black_scholes_put`**, and **`EuropeanPricer`**. Compare **`MoneyEstimate.mean.amount`** to analytical prices.

In [1]:
import math
from finstack_quant.monte_carlo import black_scholes_call, black_scholes_put, EuropeanPricer

S, K, r, q, vol, T = 100.0, 100.0, 0.05, 0.02, 0.25, 1.0
c = black_scholes_call(S, K, r, q, vol, T)
p = black_scholes_put(S, K, r, q, vol, T)
print(f"BS call: {c:.8f}")
print(f"BS put:  {p:.8f}")

forward = S * math.exp((r - q) * T)
parity_lhs = c - p
parity_rhs = math.exp(-r * T) * (forward - K)
print(f"Put-call parity LHS (C-P): {parity_lhs:.8f}")
print(f"Put-call parity RHS:       {parity_rhs:.8f}")
print(f"Parity residual:           {abs(parity_lhs - parity_rhs):.2e}")


BS call: 11.12376193
BS put:  8.22683705
Put-call parity LHS (C-P): 2.89692488
Put-call parity RHS:       2.89692488
Parity residual:           2.53e-14


## Examples

**Convergence study**: increase **`num_paths`** and tabulate **absolute error** versus **`black_scholes_call`**.

**Antithetic variates**: pair $Z$ with $-Z$ in a one-step terminal-spot estimator (see below).

**Greeks (FD)**: central differences on **`black_scholes_call`** for **spot** (delta) and **vol** (vega).

In [2]:
from finstack_quant.monte_carlo import black_scholes_call, EuropeanPricer

S, K, r, q, vol, T = 100.0, 100.0, 0.05, 0.0, 0.20, 1.0
analytical = black_scholes_call(S, K, r, q, vol, T)
print(f"Analytical ATM call: {analytical:.8f}")
counts = [1000, 5000, 10_000, 50_000, 100_000]
print("num_paths | MC_mean | stderr | |err|")
for n in counts:
    est = EuropeanPricer(num_paths=n, seed=123).price_call(S, K, r, q, vol, T, num_steps=252)
    m = est.mean.amount
    print(f"{n:9d} | {m:.8f} | {est.stderr:.8f} | {abs(m - analytical):.8f}")


Analytical ATM call: 10.45058357
num_paths | MC_mean | stderr | |err|
     1000 | 10.81611404 | 0.47076579 | 0.36553046
     5000 | 10.39422305 | 0.20740933 | 0.05636053
    10000 | 10.46263677 | 0.14602496 | 0.01205320
    50000 | 10.50791343 | 0.06601823 | 0.05732986
   100000 | 10.42770870 | 0.04657890 | 0.02287487


### Antithetic variates (variance reduction)

Pair each standard normal draw $Z$ with $-Z$ in the one-step GBM terminal-spot formula. For a **European call** (monotone in $Z$), averaging the pair typically **cuts variance** versus the same number of **independent** draws, so the estimate often lands closer to Black–Scholes for a fixed random budget.

In [3]:
import math
import random

from finstack_quant.monte_carlo import black_scholes_call

S, K, r, q, vol, T = 100.0, 100.0, 0.05, 0.0, 0.20, 1.0
truth = black_scholes_call(S, K, r, q, vol, T)
n_pairs = 2_000
rng = random.Random(42)


def terminal_spot(z: float) -> float:
    return S * math.exp((r - q - 0.5 * vol * vol) * T + vol * math.sqrt(T) * z)


disc = math.exp(-r * T)
crude_sum = 0.0
anti_sum = 0.0
for _ in range(n_pairs):
    z = rng.gauss(0.0, 1.0)
    crude_sum += disc * max(terminal_spot(z) - K, 0.0)
    anti_sum += disc * 0.5 * (
        max(terminal_spot(z) - K, 0.0) + max(terminal_spot(-z) - K, 0.0)
    )

crude = crude_sum / n_pairs
anti = anti_sum / n_pairs
print(f"Black–Scholes (exact):     {truth:.8f}")
print(f"Crude MC ({n_pairs} draws):     {crude:.8f}  |error|={abs(crude - truth):.8f}")
print(
    f"Antithetic ({n_pairs} pairs): {anti:.8f}  |error|={abs(anti - truth):.8f}  "
    "(2 terminal spots per outer loop)"
)

Black–Scholes (exact):     10.45058357
Crude MC (2000 draws):     10.29696152  |error|=0.15362205
Antithetic (2000 pairs): 10.51326135  |error|=0.06267778  (2 terminal spots per outer loop)


In [4]:
from finstack_quant.monte_carlo import black_scholes_call

S, K, r, q, vol, T = 100.0, 100.0, 0.05, 0.0, 0.20, 1.0
h_s = 1.0
delta = (
    black_scholes_call(S + h_s, K, r, q, vol, T)
    - black_scholes_call(S - h_s, K, r, q, vol, T)
) / (2 * h_s)
print(f"Finite-difference delta (central, bump spot={h_s}): {delta:.8f}")

h_v = 0.01
vega = (
    black_scholes_call(S, K, r, q, vol + h_v, T)
    - black_scholes_call(S, K, r, q, vol - h_v, T)
) / (2 * h_v)
print(f"Finite-difference vega (central, bump vol={h_v}): {vega:.8f}")
print("(Vega here is per absolute vol bump; scale to 'per 1% vol' by dividing by 100 if needed.)")


Finite-difference delta (central, bump spot=1.0): 0.63674469
Finite-difference vega (central, bump vol=0.01): 37.52098306
(Vega here is per absolute vol bump; scale to 'per 1% vol' by dividing by 100 if needed.)


### Monte Carlo finite-difference Greeks (with common random numbers)

`finite_diff_delta` / `finite_diff_gamma` bump-and-revalue under fresh randoms (noisy), while the `_crn` variants reuse **common random numbers** to slash the differencing variance. Each returns `(estimate, stderr)`.

In [5]:
from finstack_quant.monte_carlo import (
    finite_diff_delta,
    finite_diff_delta_crn,
    finite_diff_gamma,
    finite_diff_gamma_crn,
)

kw = dict(spot=100.0, strike=100.0, rate=0.05, div_yield=0.0, vol=0.20, expiry=1.0,
          num_paths=40_000, seed=7, option_type="call")
d, d_se = finite_diff_delta(**kw)
dc, dc_se = finite_diff_delta_crn(**kw)
g, g_se = finite_diff_gamma(**kw)
gc, gc_se = finite_diff_gamma_crn(**kw)
print(f"delta      = {d:.5f}  (stderr={d_se:.5f})")
print(f"delta_crn  = {dc:.5f}  (stderr={dc_se:.5f})  <- CRN cuts the stderr sharply")
print(f"gamma      = {g:.5f}  (stderr={g_se:.5f})")
print(f"gamma_crn  = {gc:.5f}  (stderr={gc_se:.5f})")

delta      = 0.63545  (stderr=0.05163)
delta_crn  = 0.63545  (stderr=0.00285)  <- CRN cuts the stderr sharply
gamma      = 0.01875  (stderr=0.17883)
gamma_crn  = 0.01875  (stderr=0.00054)


## Takeaways

- **Analytical BS** is the first sanity check for **`EuropeanPricer`**.
- **Put–call parity** catches discounting or input mismatches instantly.
- **Error vs paths** illustrates **\( 1/\sqrt{N} \)** behavior of standard errors.
- **Antithetic** pairing of normals is a simple **variance reduction** technique for smooth payoffs (e.g. European vanillas).
- **Bumping** is simple and robust for teaching; production often prefers **pathwise** or **likelihood ratio** Greeks for MC.